In [ ]:
import pandas as pd
df_train=pd.read_csv('data/train_fe_cat.csv')
df_test=pd.read_csv('data/test_fe_cat.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [3]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [4]:
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [5]:
df_test=df_test.drop(['Driver'],axis=1)
df_train_merged=df_train_merged.drop(['Driver'],axis=1)
df_train=df_train.drop(['Driver'],axis=1)


In [6]:
# predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc').fit(
#     train_data=df_train,
#     ag_args_fit={"num_gpus": 2},
#     time_limit=3600*9,
#     presets='best_quality',
#     verbosity=3,
#     num_stack_levels=0
# )

In [7]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [ ]:
import os
import shutil
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models4"
TIME_LIMIT = 30 * 60  # 30 minutes (1800 seconds) per model family

models = {
    "GBM": [
        {},  # Generates LightGBM_BAG_L1
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # Generates LightGBMLarge_BAG_L1
    ],
    "XGB": {},  # Generates XGBoost_BAG_L1
    "CAT": {},  # Generates CatBoost_BAG_L1
}

# ============================================================
# DELETE OLD MODELS
# ============================================================

# if os.path.exists(MODEL_DIR):
#     print(f"Deleting existing model directory: {MODEL_DIR}")
#     shutil.rmtree(MODEL_DIR)

# os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Fresh model directory created: {MODEL_DIR}")

# ============================================================
# TRAIN ONE MODEL AT A TIME
# ============================================================

predictors = {}
results = []

for model_name, model_config in models.items():

    model_path = os.path.join(MODEL_DIR, model_name)

    print("\n" + "=" * 80)
    print(f"TRAINING: {model_name}")
    print(f"TIME LIMIT: 30 MINUTES")
    print(f"MODEL PATH: {model_path}")
    print("=" * 80)

    predictor = TabularPredictor(
        label=TARGET,
        eval_metric='roc_auc',
        path=model_path
    ).fit(
        train_data=df_train,
        presets="best_quality",  # Enables bagging (_BAG_L1) and out-of-fold validation
        hyperparameters={model_name: model_config},  # Maps key to config dict properly
        ag_args_fit={
            'num_gpus': 1
        },
        time_limit=TIME_LIMIT,
        verbosity=3,
        num_stack_levels=0
    )

    predictors[model_name] = predictor

    # ========================================================
    # GET RESULT
    # ========================================================

    leaderboard = predictor.leaderboard(silent=True)
    best_row = leaderboard.iloc[0]

    results.append({
        'Model Family': model_name,
        'Best AutoGluon Model': best_row['model'],
        'Validation ROC-AUC': best_row['score_val'],
        'Fit Time (sec)': best_row['fit_time'],
        'Predict Time (sec)': best_row['pred_time_val']
    })

    print(f"\n{model_name} completed.")
    print(f"Validation ROC-AUC: {best_row['score_val']:.6f}")

# ============================================================
# FINAL COMPARISON
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    'Validation ROC-AUC',
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(results_df)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.13.0+cu126
CUDA Version:       12.6
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.03 GB / 15.06 GB (26.7%)
Disk Space Avail:   711.20 GB / 930.47 GB (76.4%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 

Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "c:\Darshak\Projects\Hackathon\ag_models4\GBM"
Train Data Rows:    439140
Train Data Columns: 57
Label Column:       PitNextLap
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [np.float64(1.0), np.float64(0.0)]
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       binary
Preprocessing data...


Deleting existing model directory: ./ag_models4
Fresh model directory created: ./ag_models4

TRAINING: GBM
TIME LIMIT: 30 MINUTES
MODEL PATH: ./ag_models4\GBM


Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    4040.40 MB
	Train Data (Original)  Memory Usage: 235.80 MB (5.8% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 3 features to boolean dtype as they only contain 2 unique values.
			Original Features (exact raw dtype, raw dtype):
				('float64', 'float') : 21 | ['TyreLife', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', ...]
				('int64', 'int')     : 25 | ['Unnamed: 0', 'Year', 'PitStop', 'LapNumber', 'Stint', ...]
				('object', 'object') :  2 | ['Compound', 'Race']
			Types of features in original data (raw dtype, special dtypes):
				('float', [])  : 21 | ['TyreLife', 'LapTime (s)',

[50]	valid_set's binary_logloss: 0.268532
[100]	valid_set's binary_logloss: 0.249238
[150]	valid_set's binary_logloss: 0.242833
[200]	valid_set's binary_logloss: 0.239158
[250]	valid_set's binary_logloss: 0.237178
[300]	valid_set's binary_logloss: 0.236075
[350]	valid_set's binary_logloss: 0.235218
[400]	valid_set's binary_logloss: 0.234336
[450]	valid_set's binary_logloss: 0.233665
[500]	valid_set's binary_logloss: 0.232995
[550]	valid_set's binary_logloss: 0.232264
[600]	valid_set's binary_logloss: 0.231744
[650]	valid_set's binary_logloss: 0.231349
[700]	valid_set's binary_logloss: 0.230934
[750]	valid_set's binary_logloss: 0.230608
[800]	valid_set's binary_logloss: 0.230224
[850]	valid_set's binary_logloss: 0.230109
[900]	valid_set's binary_logloss: 0.229942
[950]	valid_set's binary_logloss: 0.229806
[1000]	valid_set's binary_logloss: 0.229732
[1050]	valid_set's binary_logloss: 0.229563
[1100]	valid_set's binary_logloss: 0.229446
[1150]	valid_set's binary_logloss: 0.22939
[1200]	va

...

	Fit constraints were validated upstream on the full training data, skipping...
	Training S1F8 with GPU, note that this may negatively impact model quality compared to CPU training.
	Catboost model hyperparameters: {'iterations': 10000, 'learning_rate': 0.05, 'allow_writing_files': False, 'eval_metric': 'Logloss', 'random_seed': 0, 'thread_count': 16, 'task_type': 'GPU'}


0:	learn: 0.6297417	test: 0.6297829	best: 0.6297829 (0)	total: 10.6ms	remaining: 10.6ms
1:	learn: 0.5774066	test: 0.5774634	best: 0.5774634 (1)	total: 24.9ms	remaining: 0us
bestTest = 0.5774634381
bestIteration = 1
0:	learn: 0.6297339	test: 0.6297751	best: 0.6297751 (0)	total: 18ms	remaining: 37.6s
20:	learn: 0.3208769	test: 0.3203010	best: 0.3203010 (20)	total: 347ms	remaining: 34.1s
40:	learn: 0.2881425	test: 0.2874317	best: 0.2874317 (40)	total: 693ms	remaining: 34.6s
60:	learn: 0.2770586	test: 0.2763410	best: 0.2763410 (60)	total: 1.06s	remaining: 35.2s
80:	learn: 0.2700832	test: 0.2694712	best: 0.2694712 (80)	total: 1.38s	remaining: 34.1s
100:	learn: 0.2658269	test: 0.2653005	best: 0.2653005 (100)	total: 1.68s	remaining: 33s
120:	learn: 0.2616686	test: 0.2612385	best: 0.2612385 (120)	total: 1.98s	remaining: 32.1s
140:	learn: 0.2585588	test: 0.2583064	best: 0.2583064 (140)	total: 2.27s	remaining: 31.4s
160:	learn: 0.2563238	test: 0.2562308	best: 0.2562308 (160)	total: 2.59s	remaini

Saving c:\Darshak\Projects\Hackathon\ag_models4\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models4\CAT\models\CatBoost_BAG_L1\model.pkl
	0.9446	 = Validation score   (roc_auc)
	157.69s	 = Training   runtime
	0.14s	 = Validation runtime
	387764.6	 = Inference  throughput (rows/s | 54893 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models4\CAT\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models4\CAT\models\CatBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.greedy_ensemble.greedy_weighted_ensemble_model.GreedyWeightedEnsembleModel'>, 'priority': 0}, 'ag_args_ensemble': {'save_bag_folds': True}}
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 1640.52s of remaining time.
	


CAT completed.
Validation ROC-AUC: 0.944636

FINAL MODEL COMPARISON


,Model Family,Best AutoGluon Model,Validation ROC-AUC,Fit Time (sec),Predict Time (sec)
0,GBM,WeightedEnsemble_L2,0.948897,257.058052,11.228144
1,XGB,XGBoost_BAG_L1,0.947970,128.010609,1.460929
2,CAT,CatBoost_BAG_L1,0.944636,157.690471,0.141563


In [16]:
import os
import pandas as pd
from autogluon.tabular import TabularPredictor

# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = "./ag_models3"

all_leaderboards = []

# ============================================================
# SCAN & LOAD ALL TRAINED PREDICTORS
# ============================================================

if os.path.exists(MODEL_DIR):
    # Check both subdirectories and root directory for saved predictors
    folder_candidates = [MODEL_DIR] + [
        os.path.join(MODEL_DIR, d) 
        for d in os.listdir(MODEL_DIR) 
        if os.path.isdir(os.path.join(MODEL_DIR, d))
    ]

    for folder_path in folder_candidates:
        # Check if directory contains a valid predictor file
        if os.path.exists(os.path.join(folder_path, "predictor.pkl")):
            try:
                folder_name = os.path.basename(folder_path)
                print(f"Loading predictor from: {folder_path}")
                
                # Load predictor from disk
                predictor = TabularPredictor.load(folder_path)
                
                # Fetch leaderboard
                lb = predictor.leaderboard(silent=True)
                lb.insert(0, "folder_name", folder_name)
                
                all_leaderboards.append(lb)
            except Exception as e:
                print(f"Failed to load predictor from {folder_path}: {e}")

# ============================================================
# MERGE AND PRESENT COMBINED LEADERBOARD
# ============================================================

if all_leaderboards:
    combined_leaderboard = pd.concat(all_leaderboards, ignore_index=True)

    # Sort all trained models by validation ROC-AUC score descending
    combined_leaderboard = combined_leaderboard.sort_values(
        by="score_val", ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 80)
    print("COMBINED LEADERBOARD OF ALL LOADED MODELS")
    print("=" * 80)

    display(combined_leaderboard)
else:
    print(f"No valid AutoGluon predictors (`predictor.pkl`) found under '{MODEL_DIR}'.")

Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\CAT\models\trainer.pkl


Loading predictor from: ./ag_models3\CAT
Loading predictor from: ./ag_models3\GBM


Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\GBM\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\trainer.pkl


Loading predictor from: ./ag_models3\XGB

COMBINED LEADERBOARD OF ALL LOADED MODELS


,folder_name,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,GBM,WeightedEnsemble_L2,0.957781,roc_auc,36.298102,410.961213,0.051081,2.553276,2,True,3
1,GBM,LightGBM_BAG_L1,0.957254,roc_auc,12.108692,179.215388,12.108692,179.215388,1,True,1
2,XGB,XGBoost_BAG_L1,0.956154,roc_auc,3.170212,255.022554,3.170212,255.022554,1,True,1
3,XGB,WeightedEnsemble_L2,0.956154,roc_auc,3.220778,255.079135,0.050566,0.056581,2,True,2
4,GBM,LightGBMLarge_BAG_L1,0.955598,roc_auc,24.138329,229.192549,24.138329,229.192549,1,True,2
5,CAT,WeightedEnsemble_L2,0.943546,roc_auc,0.214443,181.873038,0.050315,0.056394,2,True,2
6,CAT,CatBoost_BAG_L1,0.943546,roc_auc,0.164128,181.816644,0.164128,181.816644,1,True,1


In [17]:
df=predictor.predict_proba(df_test)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\WeightedEnsemble_L2\model.pkl


,0,1
0,0.999910,0.000090
1,0.999970,0.000030
2,0.999918,0.000082
3,0.978575,0.021425
4,0.998234,0.001766


In [18]:
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [19]:
df_sample_out['PitNextLap']=df[1]

In [20]:
df_sample_out.head()

,id,PitNextLap
0,439140,0.000090
1,439141,0.000030
2,439142,0.000082
3,439143,0.021425
4,439144,0.001766


In [21]:
df_sample_out.to_csv("My_output/feature_engineered_submission.csv")

In [24]:
### My sampling testing
from multi_sampling_test import process_and_evaluate_all_csvs
data_dict = process_and_evaluate_all_csvs(predictor,folder_path="Sampling_data_to_test")

Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\XGBoost_BAG_L1\model.pkl



----------------------------------------
 Processing: sample_part_1_10000k.csv
----------------------------------------


Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\XGBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_1_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9403
Mismatches                         : 597
Accuracy (%)                       : 94.03
True Positives (Actual 1, Pred 1)  : 1683
True Negatives (Actual 0, Pred 0)  : 7720
False Positives (Actual 0, Pred 1) : 280
False Negatives (Actual 1, Pred 0) : 317

----------------------------------------
 Processing: sample_part_2_10000k.csv
----------------------------------------


Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\XGBoost_BAG_L1\model.pkl


...

--- Per-File Analysis [sample_part_3_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9379
Mismatches                         : 621
Accuracy (%)                       : 93.79
True Positives (Actual 1, Pred 1)  : 2133
True Negatives (Actual 0, Pred 0)  : 7246
False Positives (Actual 0, Pred 1) : 254
False Negatives (Actual 1, Pred 0) : 367

----------------------------------------
 Processing: sample_part_4_10000k.csv
----------------------------------------


Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\XGBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_4_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9344
Mismatches                         : 656
Accuracy (%)                       : 93.44
True Positives (Actual 1, Pred 1)  : 2574
True Negatives (Actual 0, Pred 0)  : 6770
False Positives (Actual 0, Pred 1) : 230
False Negatives (Actual 1, Pred 0) : 426

----------------------------------------
 Processing: sample_part_5_10000k.csv
----------------------------------------


Loading: c:\Darshak\Projects\Hackathon\ag_models3\XGB\models\WeightedEnsemble_L2\model.pkl


--- Per-File Analysis [sample_part_5_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9518
Mismatches                         : 482
Accuracy (%)                       : 95.18
True Positives (Actual 1, Pred 1)  : 833
True Negatives (Actual 0, Pred 0)  : 8685
False Positives (Actual 0, Pred 1) : 315
False Negatives (Actual 1, Pred 0) : 167

      OVERALL CUMULATIVE ANALYSIS REPORT        
      PREDICTION ANALYSIS REPORT        
Total Records                      : 50000
Correct Matches                    : 47101
Mismatches                         : 2899
Accuracy (%)                       : 94.2
True Positives (Actual 1, Pred 1)  : 8502
True Negatives (Actual 0, Pred 0)  : 38599
False Positives (Actual 0, Pred 1) : 1401
False Negatives (Actual 1, Pred 0) : 1498
